# Homework 1: Run an LLM on Your Laptop

In class you used cloud APIs (OpenAI, OpenRouter). Now you'll run a model **locally** using [Ollama](https://ollama.com).

**Before you start:** make sure you have:
1. Installed Ollama from [ollama.com/download](https://ollama.com/download)
2. Pulled a model: `ollama pull llama3.2`
3. Ollama is running (it starts automatically after install; check with `ollama list`)

## Setup

The key insight: **Ollama exposes an OpenAI-compatible API** at `http://localhost:11434/v1`.

This means the same `openai` Python SDK you used in class works with local models — just change the `base_url`.

In [ ]:
import time

from openai import OpenAI

# Connect to your local Ollama server
local_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",  # Required by the SDK, but Ollama ignores it
)

# Quick check: list available models
models = local_client.models.list()
print("Available local models:")
for m in models.data:
    print(f"  - {m.id}")

## Exercise 1: Your first local chat

Use the same prompt from class: *"Explain Software 3.0 in one sentence."*

Compare with what you got from the cloud API.

In [ ]:
MODEL = "llama3.2"  # Change this if you pulled a different model

start = time.time()

response = local_client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Explain Software 3.0 in one sentence."}
    ],
    temperature=0.7,
)

elapsed = time.time() - start

print(response.choices[0].message.content)
print(f"\nTime: {elapsed:.2f}s")
if response.usage:
    print(f"Tokens: {response.usage.total_tokens}")

## Exercise 2: Cloud vs Local — side by side

Let's compare the same prompt on both backends. You'll need your OpenAI API key in `.env` for the cloud call.

**Things to notice:**
- Response quality: is the local model's answer as good?
- Speed: which is faster? (It depends on your hardware and internet connection)
- Cost: the local call is free

In [ ]:
from dotenv import load_dotenv

load_dotenv()

cloud_client = OpenAI()  # Uses OPENAI_API_KEY from .env

prompt = "What are three benefits of open-source software?"

# --- Cloud ---
start = time.time()
cloud_resp = cloud_client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.7,
)
cloud_time = time.time() - start

# --- Local ---
start = time.time()
local_resp = local_client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": prompt}],
    temperature=0.7,
)
local_time = time.time() - start

print("=" * 60)
print(f"CLOUD (gpt-4.1-mini) — {cloud_time:.2f}s")
print("=" * 60)
print(cloud_resp.choices[0].message.content)

print()
print("=" * 60)
print(f"LOCAL ({MODEL}) — {local_time:.2f}s")
print("=" * 60)
print(local_resp.choices[0].message.content)

## Exercise 3: Streaming from a local model

Streaming works the same way as with the cloud API. Measure time-to-first-token (TTFT) — it's often much lower with local models since there's no network latency.

In [ ]:
start = time.time()
first_token_at = None

stream = local_client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Write a haiku about running AI locally."}],
    stream=True,
)

print("Response: ", end="", flush=True)
for chunk in stream:
    content = chunk.choices[0].delta.content
    if not content:
        continue
    if first_token_at is None:
        first_token_at = time.time()
    print(content, end="", flush=True)

total = time.time() - start
ttft = (first_token_at - start) if first_token_at else None

print("\n")
print(f"TTFT:  {ttft:.2f}s" if ttft else "TTFT: n/a")
print(f"Total: {total:.2f}s")

## Exercise 4: System prompts and personas

Try giving the local model a system prompt. This works exactly like with OpenAI.

**Your task:** try at least 2 different system prompts and see how the model's behavior changes.

In [ ]:
system_prompts = [
    "You are a pirate. Answer everything in pirate speak.",
    "You are a Socratic tutor. Never give the answer directly — only ask guiding questions.",
]

user_question = "What is machine learning?"

for i, system_prompt in enumerate(system_prompts, 1):
    response = local_client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_question},
        ],
        temperature=0.7,
    )
    print(f"--- System prompt {i}: {system_prompt[:50]}... ---")
    print(response.choices[0].message.content)
    print()

## Exercise 5: Try a different model

Pull another model and compare. In a terminal, run:

```bash
ollama pull gemma3:4b    # or qwen3:4b, or llama3.2:1b
```

Then run the same prompt on both models. Which gives better answers? Which is faster?

In [ ]:
# TODO: change SECOND_MODEL to the model you pulled
SECOND_MODEL = "gemma3:4b"  # Change this!

prompt = "Explain the difference between AI, machine learning, and deep learning in 3 sentences."

for model_name in [MODEL, SECOND_MODEL]:
    start = time.time()
    resp = local_client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
    )
    elapsed = time.time() - start
    print(f"--- {model_name} ({elapsed:.2f}s) ---")
    print(resp.choices[0].message.content)
    print()

## Reflection

Write a few sentences about what you observed. Consider:

1. **Quality gap**: how did local model responses compare to GPT-4.1-mini?
2. **Speed**: was local faster or slower? Did TTFT differ from total time?
3. **When would you choose local vs cloud?** Think about privacy, cost, latency, and quality.
4. **One SDK, three providers**: you've now used the `openai` SDK with OpenAI, OpenRouter, and Ollama. What's the advantage of this pattern?

*Write your reflection here.*